In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer, AerSimulator
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime.fake_provider import FakeWashingtonV2

from matplotlib import pyplot as plt
import sys
import numpy as np
from qiskit.quantum_info import Pauli, Statevector
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.visualization import plot_gate_map
from qiskit.transpiler import CouplingMap

sys.path.append("pauli_lindblad_per/")
from tomography.experiment import SparsePauliTomographyExperiment as tomography

plt.style.use("ggplot")

In [ ]:
backend = FakeWashingtonV2()
plot_gate_map(backend)
"""coupling_list = backend.configuration().coupling_map
print("Coupling Map:", coupling_list)"""

noise_model = NoiseModel.from_backend(backend)
simulator = AerSimulator.from_backend(backend)


In [ ]:
properties = backend.properties()

for qubit in range(backend.num_qubits):
    t1 = properties.t1(qubit)
    t2 = properties.t2(qubit)
    readout_err = properties.readout_error(qubit)
    print(f"Qubit {qubit}: T1 = {t1:.2e} s, T2 = {t2:.2e} s, Readout Error = {readout_err:.4f}")

In [ ]:

# 1. Circuit: GHZ-State mit 4 Qubits → Erwartungswert von XXXX = +1
n = 4
qc = QuantumCircuit(n)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)

# Idealer Erwartungswert berechnen
pauli = Pauli("XXXX")
target_value = Statevector.from_instruction(qc).expectation_value(pauli)
print(f"Idealer Erwartungswert XXXX = {target_value.real:.3f}")

inst_map = [0,1,2,3]

experiment = tomography(
    circuits = [qc], #list of circuits from which to get the layers requiring tomography
    inst_map = inst_map, #which physical qubits on the processor to map the qubits in the circuit
    backend = backend, #quantum backend
    used_qubits= [0,1,2,3]
    )

experiment.generate(
    samples = 100, #Number of samples to take from the fidelity pair measurements
    single_samples = 250, #samples to take from the pair-breaking measurements
    depths = [2,4,16,32,64] #values of 2n to make the pair measurements. Numbers must be even and non-zero
    )

shots = 1024
def executor(circuits):
    return backend.run(circuits, shots = shots, noise_model=noise_model).result().get_counts()



In [ ]:
#run the experiment
experiment.run(executor)

noisedataframe = experiment.analyze()

In [ ]:
layer = experiment.analysis.get_layer_data(0)
layer.plot_infidelitites(plot_style = 2)

In [ ]:
layer.plot_coeffs(plot_style = 2)

In [ ]:
# 2. PER-Experiment Setup
perexp = experiment.create_per_experiment([qc])

perexp.generate(
    expectations=["XXXX"],
    samples=500,
    noise_strengths=[0, 1, 2]
)

# 3. Ausführen
perexp.run(executor)

# 4. Analysieren
per_results = perexp.analyze()
result_xxxx = per_results[0].get_result("XXXX")


In [ ]:
result_xxxx.plot()

In [ ]:
qc.draw('mpl')